# Time Series Forecasting — Colab Demo

Benchmark for **TimesNet, TimeMixer, PatchTST and ModernTCN** on ETT datasets.

| Runtime | ETTh1 · pred=96 · 10 epochs |
|---------|------------------------------|
| T4 GPU  | ~10–20 min                   |
| CPU     | ~4–8 h                       |

> **Tip:** Runtime → Change runtime type → T4 GPU

In [1]:
!nvidia-smi

Wed May 20 12:19:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

True
Tesla T4


In [3]:
!git clone https://github.com/caltinuzengi/Time-Series-Library.git
%cd Time-Series-Library/

Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 146 (delta 59), reused 114 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 183.44 KiB | 469.00 KiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/Time-Series-Library


In [4]:
!git pull
!pip install uv -q
!uv sync

remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 48 (delta 25), reused 48 (delta 25), pack-reused 0 (from 0)
Unpacking objects: 100% (48/48), 28.49 KiB | 1.68 MiB/s, done.
From https://github.com/caltinuzengi/Time-Series-Library
   33d94f0..8bb319a  main       -> origin/main
Updating 33d94f0..8bb319a
Fast-forward
 data_provider/data_factory.py              |   5 +
 data_provider/data_loader.py               |  12 +-
 exp/exp_anomaly.py                         | 219 ++++++++++++++++++++
 exp/exp_base.py                            |   2 +
 exp/exp_forecasting.py                     |  11 +-
 layers/ModernTCNBlock.py                   | 274 +++++++++++++++++++++++++
 models/ModernTCN.py                        | 233 +++++++++++++++++++++
 models/PatchTST.py                         |  48 +++++
 models/TimeMixer.py                        |  58 ++++++
 models/TimesNet.py                         

In [5]:
!uv run download-data

ETT Dataset Downloader
Source : github.com/zhouhaoyi/ETDataset (Informer AAAI 2021)
Dest   : /content/Time-Series-Library/data

[ETTh1]
  Source: https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv
  [████████████████████] 100%  (2536 / 2529 KB)
  ✓ Saved → data/ETTh1.csv  (2529 KB, 0.1s)
  ✓ ETTh1 verified: shape=(17420, 8), columns=['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

[ETTh2]
  Source: https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv
  [████████████████████] 100%  (2368 / 2361 KB)
  ✓ Saved → data/ETTh2.csv  (2361 KB, 0.3s)
  ✓ ETTh2 verified: shape=(17420, 8), columns=['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

[ETTm1]
  Source: https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv
  [████████████████████] 100%  (10120 / 10118 KB)
  ✓ Saved → data/ETTm1.csv  (10118 KB, 0.9s)
  ✓ ETTm1 verified: shape=(69680, 8), columns=['date', 'HUFL', 'HULL', 'MUFL', 'MU

In [6]:
!bash scripts/TimesNet/debug_ETTh1.sh

Device : cuda:0
Model  : TimesNet
Data   : ETTh1  seq=96  pred=96
2026-05-20 12:23:32.379 | INFO     | exp.exp_forecasting:_build_model:80 - Model built: TimesNet | params=9,394,005
2026-05-20 12:23:32.667 | INFO     | exp.exp_forecasting:train:130 - Loading datasets …
2026-05-20 12:23:32.820 | INFO     | exp.exp_forecasting:train:133 - Datasets loaded in 0.2s
2026-05-20 12:23:32.821 | INFO     | exp.exp_forecasting:_log_device_info:92 - GPU : Tesla T4 | 15.6 GB total
2026-05-20 12:23:32.821 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Train split: 8,449 samples | 529 batches (batch_size=16)
2026-05-20 12:23:32.821 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Val   split: 2,785 samples | 175 batches (batch_size=16)
2026-05-20 12:23:37.279 | INFO     | exp.exp_forecasting:train:145 - Training TimesNet on ETTh1 | pred_len=96 | epochs=1 | lr=1.00e-04 | device=cuda:0
2026-05-20 12:23:37.279 | INFO     | exp.exp_forecasting:train:150 - -----------------------------------

In [7]:
!bash scripts/TimeMixer/debug_ETTh1.sh

Device : cuda:0
Model  : TimeMixer
Data   : ETTh1  seq=96  pred=96
2026-05-20 12:28:04.361 | INFO     | exp.exp_forecasting:_build_model:80 - Model built: TimeMixer | params=75,465
2026-05-20 12:28:04.572 | INFO     | exp.exp_forecasting:train:130 - Loading datasets …
2026-05-20 12:28:04.687 | INFO     | exp.exp_forecasting:train:133 - Datasets loaded in 0.1s
2026-05-20 12:28:04.687 | INFO     | exp.exp_forecasting:_log_device_info:92 - GPU : Tesla T4 | 15.6 GB total
2026-05-20 12:28:04.687 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Train split: 8,449 samples | 529 batches (batch_size=16)
2026-05-20 12:28:04.687 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Val   split: 2,785 samples | 175 batches (batch_size=16)
2026-05-20 12:28:06.076 | INFO     | exp.exp_forecasting:train:145 - Training TimeMixer on ETTh1 | pred_len=96 | epochs=1 | lr=1.00e-02 | device=cuda:0
2026-05-20 12:28:06.076 | INFO     | exp.exp_forecasting:train:150 - -----------------------------------

In [8]:
!bash scripts/PatchTST/debug_ETTh1.sh

Device : cuda:0
Model  : PatchTST
Data   : ETTh1  seq=96  pred=96
2026-05-20 12:28:41.074 | INFO     | exp.exp_forecasting:_build_model:80 - Model built: PatchTST | params=696,398
2026-05-20 12:28:41.287 | INFO     | exp.exp_forecasting:train:130 - Loading datasets …
2026-05-20 12:28:41.400 | INFO     | exp.exp_forecasting:train:133 - Datasets loaded in 0.1s
2026-05-20 12:28:41.401 | INFO     | exp.exp_forecasting:_log_device_info:92 - GPU : Tesla T4 | 15.6 GB total
2026-05-20 12:28:41.401 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Train split: 8,449 samples | 133 batches (batch_size=64)
2026-05-20 12:28:41.401 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Val   split: 2,785 samples | 44 batches (batch_size=64)
2026-05-20 12:28:42.402 | INFO     | exp.exp_forecasting:train:145 - Training PatchTST on ETTh1 | pred_len=96 | epochs=1 | lr=1.00e-04 | device=cuda:0
2026-05-20 12:28:42.403 | INFO     | exp.exp_forecasting:train:150 - --------------------------------------

In [9]:
!bash scripts/ModernTCN/debug_ETTh1.sh

Device : cuda:0
Model  : ModernTCN
Data   : ETTh1  seq=96  pred=96
2026-05-20 12:28:52.121 | INFO     | exp.exp_forecasting:_build_model:80 - Model built: ModernTCN | params=711,246
2026-05-20 12:28:52.320 | INFO     | exp.exp_forecasting:train:130 - Loading datasets …
2026-05-20 12:28:52.430 | INFO     | exp.exp_forecasting:train:133 - Datasets loaded in 0.1s
2026-05-20 12:28:52.430 | INFO     | exp.exp_forecasting:_log_device_info:92 - GPU : Tesla T4 | 15.6 GB total
2026-05-20 12:28:52.430 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Train split: 8,449 samples | 265 batches (batch_size=32)
2026-05-20 12:28:52.430 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Val   split: 2,785 samples | 88 batches (batch_size=32)
2026-05-20 12:28:53.472 | INFO     | exp.exp_forecasting:train:145 - Training ModernTCN on ETTh1 | pred_len=96 | epochs=1 | lr=1.00e-03 | device=cuda:0
2026-05-20 12:28:53.472 | INFO     | exp.exp_forecasting:train:150 - -----------------------------------

In [ ]:
!bash scripts/TimesNet/benchmark_forecasting.sh

>>> TimesNet | ETTh1 | pred_len=24
Device : cuda:0
Model  : TimesNet
Data   : ETTh1  seq=96  pred=24
2026-05-20 12:30:25.670 | INFO     | exp.exp_forecasting:_build_model:80 - Model built: TimesNet | params=9,387,021
2026-05-20 12:30:25.898 | INFO     | exp.exp_forecasting:train:130 - Loading datasets …
2026-05-20 12:30:26.012 | INFO     | exp.exp_forecasting:train:133 - Datasets loaded in 0.1s
2026-05-20 12:30:26.012 | INFO     | exp.exp_forecasting:_log_device_info:92 - GPU : Tesla T4 | 15.6 GB total
2026-05-20 12:30:26.013 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Train split: 8,521 samples | 267 batches (batch_size=32)
2026-05-20 12:30:26.013 | INFO     | exp.exp_forecasting:_log_loader_info:99 - Val   split: 2,857 samples | 90 batches (batch_size=32)
2026-05-20 12:30:27.045 | INFO     | exp.exp_forecasting:train:145 - Training TimesNet on ETTh1 | pred_len=24 | epochs=10 | lr=1.00e-04 | device=cuda:0
2026-05-20 12:30:27.045 | INFO     | exp.exp_forecasting:train:150 - 

Epoch 001/10 [train]:  58% 620/1077 [06:19<04:15,  1.79batch/s, loss=0.1537]

: 

In [ ]:
# This cell for ETTms' of TimesNET

In [ ]:
!bash scripts/TimeMixer/benchmark_forecasting.sh

In [ ]:
!bash scripts/PatchTST/benchmark_forecasting.sh

In [ ]:
!bash scripts/ModernTCN/benchmark_forecasting.sh

## 📊 Sonuçlar

Aşağıdaki hücreler `logs/` klasöründeki JSON dosyalarını okur — önce yukarıdaki eğitim hücrelerini çalıştırın.

In [ ]:
import json, glob
import matplotlib.pyplot as plt

models    = ["TimesNet", "TimeMixer", "PatchTST", "ModernTCN"]
pred_lens = [24, 96]
dataset   = "ETTh1"   # temsili veri seti

fig, axes = plt.subplots(len(pred_lens), len(models), figsize=(18, 7), sharey=False)

for row, pred_len in enumerate(pred_lens):
    for col, model in enumerate(models):
        ax = axes[row][col]
        files = sorted(glob.glob(f"logs/{model}_{dataset}_forecasting_{pred_len}_*.json"))
        if not files:
            ax.text(0.5, 0.5, "log yok", ha="center", va="center",
                    transform=ax.transAxes, color="gray", fontsize=10)
            if row == 0:
                ax.set_title(model, fontsize=11)
            continue
        with open(files[-1]) as f:
            d = json.load(f)
        logs = d["epoch_logs"]
        epochs = [e["epoch"] for e in logs]
        ax.plot(epochs, [e["train_loss"] for e in logs], "o-",  label="train", ms=4, lw=1.5)
        ax.plot(epochs, [e["val_loss"]   for e in logs], "s--", label="val",   ms=4, lw=1.5)
        if row == 0:
            ax.set_title(model, fontsize=11)
        ax.set_xlabel("Epoch", fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        if col == 0:
            ax.set_ylabel(f"pred={pred_len}\nMSE", fontsize=9)

plt.suptitle(f"{dataset} — Training Curves (pred_len 24 vs 96)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import json, glob, pandas as pd

rows = []
for path in sorted(glob.glob("results/*_forecasting_*.json")):
    with open(path) as f:
        d = json.load(f)
    cfg, met = d["config"], d["metrics"]
    rows.append({
        "Model":    cfg.get("model",    "-"),
        "Data":     cfg.get("data",     "-"),
        "pred_len": cfg.get("pred_len", "-"),
        "MSE":  round(met.get("mse",  float("nan")), 4),
        "MAE":  round(met.get("mae",  float("nan")), 4),
        "RMSE": round(met.get("rmse", float("nan")), 4),
    })

if rows:
    df = pd.DataFrame(rows).sort_values(["Data", "pred_len", "MSE"])
    display(df)
else:
    print("Henüz sonuç yok — önce eğitim hücrelerini çalıştırın.")